In [3]:
# =========================================================
# Corrected SkypeType / SVM Full Notebook
# Stronger FGSM/BIM/PGD-style attacks in STANDARDIZED space
# Output CSV columns:
# model_name, Clean accuracy, attack type, epsilon,
# adversary accuracy, relative accuracy, ASR
# =========================================================

!pip -q install pandas numpy librosa scikit-learn tqdm

import os
import glob
import zipfile
import numpy as np
import pandas as pd
import librosa

from tqdm import tqdm
from numpy.fft import rfft, irfft
from IPython.display import display

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================================================
# 1) SETTINGS
# =========================================================
np.random.seed(42)

MODEL_NAME = "SkypeType_SVM"

TRAIN_ZIP = "/mnt/data/Train-20260303T162110Z-3-001.zip"
VALID_ZIP = "/mnt/data/Valid-20260303T161756Z-3-001.zip"
TEST_ZIP  = "/mnt/data/Test-20260303T161752Z-3-001.zip"

OUT_DIR = "/mnt/data/keystroke_dataset_svm"
os.makedirs(OUT_DIR, exist_ok=True)

# Keep these if your project requires the same epsilon values everywhere
FEATURE_EPSILONS = [0.005, 0.01, 0.05]
DFT_EPSILONS = [0.005, 0.01, 0.05]

# If you want visibly stronger FGSM/BIM/PGD, replace FEATURE_EPSILONS with:
# FEATURE_EPSILONS = [0.1, 0.25, 0.5, 1.0]

SR = 16000
MAX_SECONDS = 0.38
MAX_LEN = int(SR * MAX_SECONDS)

CLAMP_ASR_TO_ZERO = False

# =========================================================
# 2) DATASET HELPERS
# =========================================================
def unzip(zip_path, out_dir):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

def prepare_dataset():
    print("Unzipping dataset...")
    unzip(TRAIN_ZIP, OUT_DIR)
    unzip(VALID_ZIP, OUT_DIR)
    unzip(TEST_ZIP, OUT_DIR)

def find_split_dir(root, split_name):
    cands = glob.glob(os.path.join(root, "**", split_name), recursive=True)
    if not cands:
        raise FileNotFoundError(f"Could not find split folder '{split_name}' under {root}")
    return cands[0]

def load_labels(split_dir):
    csvs = glob.glob(os.path.join(split_dir, "**", "labels.csv"), recursive=True)
    if not csvs:
        raise FileNotFoundError(f"No labels.csv found under {split_dir}")
    df = pd.read_csv(csvs[0])
    if "filename" not in df.columns or "key" not in df.columns:
        raise ValueError(f"labels.csv must contain filename,key. Found: {df.columns.tolist()}")
    return df

def find_audio_root(split_dir):
    cands = glob.glob(os.path.join(split_dir, "**", "audio_data"), recursive=True)
    if cands:
        return cands[0]
    wavs = glob.glob(os.path.join(split_dir, "**", "*.wav"), recursive=True)
    if not wavs:
        raise FileNotFoundError(f"No wav files found under {split_dir}")
    return os.path.dirname(wavs[0])

# =========================================================
# 3) AUDIO + FEATURE HELPERS
# =========================================================
def wav_path(audio_root, fname):
    p = os.path.join(audio_root, fname)
    if os.path.exists(p):
        return p
    matches = glob.glob(os.path.join(audio_root, "**", fname), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Missing audio file: {fname}")
    return matches[0]

def load_fixed_wave(path, sr=SR, max_len=MAX_LEN):
    y, _ = librosa.load(path, sr=sr)
    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))
    else:
        y = y[:max_len]
    return y.astype(np.float32)

def mfcc_features(path, sr=SR, max_len=MAX_LEN, n_mfcc=20):
    y = load_fixed_wave(path, sr=sr, max_len=max_len)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    feat = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])
    return feat.astype(np.float32)

def mfcc_features_from_wave(y, sr=SR, n_mfcc=20):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    feat = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])
    return feat.astype(np.float32)

def build_Xy(df, audio_root, label2id):
    X, y = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        p = wav_path(audio_root, row["filename"])
        X.append(mfcc_features(p))
        y.append(label2id[row["key"]])
    return np.stack(X), np.array(y)

# =========================================================
# 4) ATTACK FUNCTIONS
# =========================================================
# NOTE:
# These are stronger SVM-compatible approximations, not true gradient attacks.
# We attack in SCALED feature space, because the SVM actually sees scaled inputs.

def fgsm_attack_scaled_features(X_scaled, epsilon):
    noise = np.sign(np.random.randn(*X_scaled.shape))
    return X_scaled + epsilon * noise

def bim_attack_scaled_features(X_scaled, epsilon=0.1, alpha=0.02, iters=5):
    X_orig = X_scaled.copy()
    X_adv = X_scaled.copy()

    for _ in range(iters):
        noise = np.sign(np.random.randn(*X_scaled.shape))
        X_adv = X_adv + alpha * noise
        perturbation = np.clip(X_adv - X_orig, -epsilon, epsilon)
        X_adv = X_orig + perturbation

    return X_adv

def pgd_attack_scaled_features(X_scaled, epsilon=0.1, alpha=0.02, iters=10):
    X_orig = X_scaled.copy()
    X_adv = X_orig + np.random.uniform(-epsilon, epsilon, X_scaled.shape)

    for _ in range(iters):
        noise = np.sign(np.random.randn(*X_scaled.shape))
        X_adv = X_adv + alpha * noise
        perturbation = np.clip(X_adv - X_orig, -epsilon, epsilon)
        X_adv = X_orig + perturbation

    return X_adv

def dft_attack_waveform(y, epsilon=0.01, band_fraction=0.25):
    Y = rfft(y)
    Y_adv = Y.copy()

    n_bins = len(Y_adv)
    start = int(n_bins * 0.10)
    end = int(n_bins * (0.10 + band_fraction))
    end = min(end, n_bins)

    real_noise = np.sign(np.random.randn(end - start))
    imag_noise = np.sign(np.random.randn(end - start))

    Y_adv[start:end] = Y_adv[start:end] + epsilon * (real_noise + 1j * imag_noise)

    y_adv = irfft(Y_adv, n=len(y)).astype(np.float32)
    y_adv = np.clip(y_adv, -1.0, 1.0)
    return y_adv

def build_dft_attacked_X(df, audio_root, label2id, epsilon, band_fraction=0.25):
    X_adv = []
    y_true = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"DFT attack eps={epsilon}"):
        p = wav_path(audio_root, row["filename"])
        y = load_fixed_wave(p)
        y_adv = dft_attack_waveform(y, epsilon=epsilon, band_fraction=band_fraction)
        feat_adv = mfcc_features_from_wave(y_adv)

        X_adv.append(feat_adv)
        y_true.append(label2id[row["key"]])

    return np.stack(X_adv), np.array(y_true)

# =========================================================
# 5) SUMMARY CSV HELPER
# =========================================================
def compute_asr(relative_accuracy, clamp_to_zero=False):
    asr = (1.0 - relative_accuracy) * 100.0
    if clamp_to_zero:
        asr = max(0.0, asr)
    return asr

def build_model_attack_summary(model_name, clean_accuracy, attack_results, include_clean=True):
    rows = []

    if include_clean:
        rows.append({
            "model_name": model_name,
            "Clean accuracy": round(float(clean_accuracy), 6),
            "attack type": "CLEAN",
            "epsilon": 0.0,
            "adversary accuracy": round(float(clean_accuracy), 6),
            "relative accuracy": 1.0,
            "ASR": 0.0
        })

    for attack_type, eps_dict in attack_results.items():
        for epsilon, adv_acc in eps_dict.items():
            relative_accuracy = adv_acc / clean_accuracy if clean_accuracy > 0 else np.nan
            asr = compute_asr(relative_accuracy, clamp_to_zero=CLAMP_ASR_TO_ZERO)

            rows.append({
                "model_name": model_name,
                "Clean accuracy": round(float(clean_accuracy), 6),
                "attack type": attack_type,
                "epsilon": round(float(epsilon), 6),
                "adversary accuracy": round(float(adv_acc), 6),
                "relative accuracy": round(float(relative_accuracy), 6),
                "ASR": round(float(asr), 6)
            })

    return pd.DataFrame(rows)

# =========================================================
# 6) PREPARE DATA
# =========================================================
prepare_dataset()

train_dir = find_split_dir(OUT_DIR, "Train")
valid_dir = find_split_dir(OUT_DIR, "Valid")
test_dir  = find_split_dir(OUT_DIR, "Test")

train_df = load_labels(train_dir)
valid_df = load_labels(valid_dir)
test_df  = load_labels(test_dir)

train_audio_root = find_audio_root(train_dir)
valid_audio_root = find_audio_root(valid_dir)
test_audio_root  = find_audio_root(test_dir)

class_names = sorted(train_df["key"].unique().tolist())
label2id = {k: i for i, k in enumerate(class_names)}

print("Classes:", class_names)

print("Extracting MFCC features...")
X_train, y_train = build_Xy(train_df, train_audio_root, label2id)
X_valid, y_valid = build_Xy(valid_df, valid_audio_root, label2id)
X_test,  y_test  = build_Xy(test_df, test_audio_root, label2id)

print("Feature shapes:", X_train.shape, X_valid.shape, X_test.shape)

# =========================================================
# 7) TRAIN MODEL
# =========================================================
clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=20, gamma="scale")
)

print("Training SVM model...")
clf.fit(X_train, y_train)

def evaluate(name, X, y):
    pred = clf.predict(X)
    acc = accuracy_score(y, pred)
    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y, pred, target_names=class_names, digits=4))
    return pred, acc

pred_valid, valid_acc = evaluate("VALID", X_valid, y_valid)
pred_test, clean_acc  = evaluate("TEST", X_test, y_test)

cm = confusion_matrix(y_test, pred_test)
cm_path = f"/mnt/data/{MODEL_NAME}_confusion_matrix.csv"
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)
print("Confusion matrix saved to:", cm_path)

# Extract fitted scaler and SVM from pipeline
scaler = clf.named_steps["standardscaler"]
svm = clf.named_steps["svc"]

# Attack in scaled feature space
X_test_scaled = scaler.transform(X_test)

# =========================================================
# 8) RUN ATTACKS
# =========================================================
attack_results = {
    "FGSM": {},
    "BIM": {},
    "PGD": {},
    "DFT": {}
}

print("\nRunning attacks on TEST dataset...")

# FGSM / BIM / PGD in scaled feature space
for eps in FEATURE_EPSILONS:
    print(f"\n========== Feature-space epsilon = {eps} ==========")

    X_fgsm_scaled = fgsm_attack_scaled_features(X_test_scaled, eps)
    pred_fgsm = svm.predict(X_fgsm_scaled)
    fgsm_acc = accuracy_score(y_test, pred_fgsm)
    attack_results["FGSM"][eps] = fgsm_acc
    print(f"FGSM Accuracy: {fgsm_acc:.4f}")

    X_bim_scaled = bim_attack_scaled_features(X_test_scaled, epsilon=eps, alpha=eps/5, iters=5)
    pred_bim = svm.predict(X_bim_scaled)
    bim_acc = accuracy_score(y_test, pred_bim)
    attack_results["BIM"][eps] = bim_acc
    print(f"BIM Accuracy:  {bim_acc:.4f}")

    X_pgd_scaled = pgd_attack_scaled_features(X_test_scaled, epsilon=eps, alpha=eps/10, iters=10)
    pred_pgd = svm.predict(X_pgd_scaled)
    pgd_acc = accuracy_score(y_test, pred_pgd)
    attack_results["PGD"][eps] = pgd_acc
    print(f"PGD Accuracy:  {pgd_acc:.4f}")

# DFT in waveform space
for eps in DFT_EPSILONS:
    print(f"\n========== DFT epsilon = {eps} ==========")

    X_dft, y_dft = build_dft_attacked_X(
        test_df,
        test_audio_root,
        label2id,
        epsilon=eps,
        band_fraction=0.25
    )
    pred_dft = clf.predict(X_dft)
    dft_acc = accuracy_score(y_dft, pred_dft)
    attack_results["DFT"][eps] = dft_acc
    print(f"DFT Accuracy:  {dft_acc:.4f}")

# =========================================================
# 9) SAVE FINAL CSV
# =========================================================
summary_df = build_model_attack_summary(
    model_name=MODEL_NAME,
    clean_accuracy=clean_acc,
    attack_results=attack_results,
    include_clean=True
)

summary_csv_path = f"/mnt/data/{MODEL_NAME}_attack_summary_blockchain.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("\nSaved final model summary CSV to:", summary_csv_path)
display(summary_df)

summary_excel_path = f"/mnt/data/{MODEL_NAME}_attack_summary_blockchain.xlsx"
summary_df.to_excel(summary_excel_path, index=False)
print("Saved Excel copy to:", summary_excel_path)

Unzipping dataset...
Classes: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 'space', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Extracting MFCC features...


100%|██████████| 1600/1600 [00:12<00:00, 126.93it/s]


Feature shapes: (5120, 40) (1280, 40) (1600, 40)
Training SVM model...

VALID Accuracy: 0.9656
              precision    recall  f1-score   support

           a     1.0000    1.0000    1.0000        48
           b     0.9792    0.9792    0.9792        48
           c     0.9787    0.9787    0.9787        47
           d     1.0000    1.0000    1.0000        48
           e     0.9787    0.9787    0.9787        47
           f     0.9583    0.9583    0.9583        48
           g     0.9778    0.9362    0.9565        47
           h     0.9038    1.0000    0.9495        47
           i     1.0000    0.9574    0.9783        47
           j     0.9375    0.9574    0.9474        47
           k     0.9184    0.9375    0.9278        48
           l     1.0000    0.9574    0.9783        47
           m     0.9778    0.9167    0.9462        48
           n     1.0000    0.9787    0.9892        47
           o     0.9583    0.9583    0.9583        48
           p     0.9787    0.9787    0.9

DFT attack eps=0.005: 100%|██████████| 1600/1600 [00:13<00:00, 118.90it/s]


DFT Accuracy:  0.9569

========== DFT epsilon = 0.01 ==========


...

DFT Accuracy:  0.9537

========== DFT epsilon = 0.05 ==========


DFT attack eps=0.05: 100%|██████████| 1600/1600 [00:13<00:00, 118.77it/s]


DFT Accuracy:  0.4062

Saved final model summary CSV to: /mnt/data/SkypeType_SVM_attack_summary_blockchain.csv


,model_name,Clean accuracy,attack type,epsilon,adversary accuracy,relative accuracy,ASR
0,SkypeType_SVM,0.9575,CLEAN,0.000,0.957500,1.000000,0.000000
1,SkypeType_SVM,0.9575,FGSM,0.005,0.957500,1.000000,0.000000
2,SkypeType_SVM,0.9575,FGSM,0.010,0.956875,0.999347,0.065274
3,SkypeType_SVM,0.9575,FGSM,0.050,0.957500,1.000000,0.000000
4,SkypeType_SVM,0.9575,BIM,0.005,0.957500,1.000000,0.000000
5,SkypeType_SVM,0.9575,BIM,0.010,0.956875,0.999347,0.065274
6,SkypeType_SVM,0.9575,BIM,0.050,0.955000,0.997389,0.261097
7,SkypeType_SVM,0.9575,PGD,0.005,0.957500,1.000000,0.000000
8,SkypeType_SVM,0.9575,PGD,0.010,0.957500,1.000000,0.000000
9,SkypeType_SVM,0.9575,PGD,0.050,0.955000,0.997389,0.261097


Saved Excel copy to: /mnt/data/SkypeType_SVM_attack_summary_blockchain.xlsx
